In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import random
import time
import itertools
from sklearn.model_selection import KFold

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Using device: cuda


In [2]:
def load_embedding_csv(path, id_col):
    df = pd.read_csv(path)

    ids = df[id_col].values

    # Keep only numeric columns
    feature_df = df.select_dtypes(include=[np.number])

    # Remove ID if it's numeric
    if id_col in feature_df.columns:
        feature_df = feature_df.drop(columns=[id_col])

    embeddings = feature_df.values.astype(np.float32)

    emb_dict = {i: emb for i, emb in zip(ids, embeddings)}
    dim = embeddings.shape[1]

    print(path, "| dim =", dim)

    return emb_dict, dim


embedding_folder = "embeddings/"

drug1, d1 = load_embedding_csv(os.path.join(embedding_folder,"drug_mol2vec.csv"), "drug_id")
drug2, d2 = load_embedding_csv(os.path.join(embedding_folder,"drug_gin.csv"), "drug_id")
drug3, d3 = load_embedding_csv(os.path.join(embedding_folder,"drug_unimol.csv"), "drug_id")

prot1, p1 = load_embedding_csv(os.path.join(embedding_folder,"protein_protvec.csv"), "protein_id")
prot2, p2 = load_embedding_csv(os.path.join(embedding_folder,"protein_protbert.csv"), "protein_id")
prot3, p3 = load_embedding_csv(os.path.join(embedding_folder,"protein_esm.csv"), "protein_id")

drug_embs_all = [drug1, drug2, drug3]
prot_embs_all = [prot1, prot2, prot3]

drug_dims = [d1, d2, d3]
prot_dims = [p1, p2, p3]

interaction_csv = r"C:\Users\Neuroscience Lab\Downloads\Test_model\Test_model\PDAC_interactions.csv"
df = pd.read_csv(interaction_csv)

embeddings/drug_mol2vec.csv | dim = 300
embeddings/drug_gin.csv | dim = 1600
embeddings/drug_unimol.csv | dim = 512
embeddings/protein_protvec.csv | dim = 100
embeddings/protein_protbert.csv | dim = 1024
embeddings/protein_esm.csv | dim = 1280


In [3]:
class DrugProteinDataset(Dataset):

    def __init__(self, df, drug_embs, prot_embs):
        self.df = df.reset_index(drop=True)
        self.drug_embs = drug_embs
        self.prot_embs = prot_embs

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        drug_feats = np.concatenate([emb[row["drug_id"]] for emb in self.drug_embs])
        prot_feats = np.concatenate([emb[row["protein_id"]] for emb in self.prot_embs])

        return (
            torch.tensor(drug_feats),
            torch.tensor(prot_feats),
            torch.tensor(row["affinity"], dtype=torch.float32)
        )

In [4]:
class AffinityModel(nn.Module):

    def __init__(self, drug_dim, prot_dim):

        super().__init__()

        self.drug_proj = nn.Sequential(
            nn.Linear(drug_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU()
        )

        self.prot_proj = nn.Sequential(
            nn.Linear(prot_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU()
        )

        self.cnn = nn.Sequential(

            nn.Conv1d(1, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, 3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.AdaptiveMaxPool1d(1)
        )

        self.regressor = nn.Sequential(

            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, drug, protein):

        drug = F.normalize(drug, p=2, dim=1)
        protein = F.normalize(protein, p=2, dim=1)

        drug = self.drug_proj(drug)
        protein = self.prot_proj(protein)

        x = torch.cat([drug, protein], dim=1)

        x = x.unsqueeze(1)

        x = self.cnn(x)

        x = x.squeeze(-1)

        out = self.regressor(x)

        return out.squeeze()

In [5]:
# Using ALL embeddings (no selection)

selected_drug_embs = drug_embs_all
selected_prot_embs = prot_embs_all

drug_dim = sum(drug_dims)
prot_dim = sum(prot_dims)

dataset = DrugProteinDataset(df, selected_drug_embs, selected_prot_embs)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

model = AffinityModel(drug_dim, prot_dim).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()
    losses = []

    for drug, prot, y in loader:
        drug, prot, y = drug.to(DEVICE), prot.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        preds = model(drug, prot)

        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {np.mean(losses):.4f}")

Epoch 1/30 | Loss: 6.9046
Epoch 2/30 | Loss: 2.9501
Epoch 3/30 | Loss: 2.7115
Epoch 4/30 | Loss: 2.6923
Epoch 5/30 | Loss: 2.6220
Epoch 6/30 | Loss: 2.7948
Epoch 7/30 | Loss: 2.6148
Epoch 8/30 | Loss: 2.4865
Epoch 9/30 | Loss: 2.6730
Epoch 10/30 | Loss: 2.5046
Epoch 11/30 | Loss: 2.5270
Epoch 12/30 | Loss: 2.4781
Epoch 13/30 | Loss: 2.4053
Epoch 14/30 | Loss: 2.4085
Epoch 15/30 | Loss: 2.4664
Epoch 16/30 | Loss: 2.5487
Epoch 17/30 | Loss: 2.4227
Epoch 18/30 | Loss: 2.4908
Epoch 19/30 | Loss: 2.7721
Epoch 20/30 | Loss: 2.4537
Epoch 21/30 | Loss: 2.3443
Epoch 22/30 | Loss: 2.3903
Epoch 23/30 | Loss: 2.3960
Epoch 24/30 | Loss: 2.4040
Epoch 25/30 | Loss: 8.4860
Epoch 26/30 | Loss: 4.0085
Epoch 27/30 | Loss: 2.8301
Epoch 28/30 | Loss: 2.4898
Epoch 29/30 | Loss: 2.6325
Epoch 30/30 | Loss: 2.4750


In [6]:
save_path = "final_model/affinity_model.pth"
os.makedirs("final_model", exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "drug_dim": drug_dim,
    "prot_dim": prot_dim,
    "drug_names": ["mol2vec", "gin", "unimol"],
    "prot_names": ["protvec", "protbert", "esm"]
}, save_path)

print(f"Model saved at {save_path}")

Model saved at final_model/affinity_model.pth


In [7]:
checkpoint = torch.load("final_model/affinity_model.pth", map_location=DEVICE)

model = AffinityModel(checkpoint["drug_dim"], checkpoint["prot_dim"]).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


def predict(drug_id, protein_id):

    drug_feats = np.concatenate([emb[drug_id] for emb in drug_embs_all])
    prot_feats = np.concatenate([emb[protein_id] for emb in prot_embs_all])

    drug_tensor = torch.tensor(drug_feats).unsqueeze(0).to(DEVICE)
    prot_tensor = torch.tensor(prot_feats).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(drug_tensor, prot_tensor).item()

    return pred

C:\Users\Neuroscience Lab\AppData\Local\Temp\ipykernel_25604\2987367139.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("final_model/affinity_mod